In [2]:
!pip install -q pesq==0.0.4 soundfile

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm
from pesq import pesq

## PATH 변경해 주세요!
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/signal_and_system/project/noise_reduction_challenge")
INPUT_DIR = DATA_ROOT / "input"
OUTPUT_DIR = DATA_ROOT / "output"
train_df = pd.read_csv(DATA_ROOT / "train" / "samples.csv")
val_df = pd.read_csv(DATA_ROOT / "validation" / "samples.csv")
print(len(train_df), len(val_df))

1296 164


In [9]:
import numpy as np
from scipy import signal
from scipy.ndimage import uniform_filter1d
from scipy.special import exp1

In [11]:
HR_SR = 16000
HR_HOP = 128
HR_FFTS = (1024, 2048)
HR_NOISE_FRAC = 0.10
HR_ALPHA_DD = 0.98
HR_GMIN1_DB = -20.0
HR_GMIN2_DB = -30.0
HR_XI_MIN_DB = -25.0
HR_RHO = 0.3
HR_HPF_HZ = 100.0

In [12]:
def _hr_stft(x, n):
    return signal.stft(x, fs=HR_SR, window="hann", nperseg=n, noverlap=n - HR_HOP, nfft=n,
                       boundary="zeros", padded=True)[2]

In [13]:
def _hr_istft(z, length, n):
    y = signal.istft(z, fs=HR_SR, window="hann", nperseg=n, noverlap=n - HR_HOP, nfft=n,
                     input_onesided=True, boundary=True)[1]
    return y[:length]

In [14]:
def _hr_noise_psd(P, n):
    edge = n // (2 * HR_HOP)
    T = P.shape[1]
    full = np.arange(edge, T - edge) if T > 2 * edge + 1 else np.arange(T)
    e = P[:, full].mean(0)
    k = max(1, int(np.ceil(HR_NOISE_FRAC * len(full))))
    sel = full[np.argsort(e, kind="stable")[:k]]
    return np.maximum(uniform_filter1d(P[:, sel].mean(1), 3, mode="nearest"), 1e-12)


In [15]:
def _hr_lmmse_gain(xi, gamma):
    v = np.maximum(xi / (1 + xi) * gamma, 1e-8)
    return xi / (1 + xi) * np.exp(0.5 * exp1(v))

In [16]:
def _hr_single(x, length, n):
    Z = _hr_stft(x, n)
    P = np.abs(Z) ** 2
    N = _hr_noise_psd(P, n)
    xi_min = 10 ** (HR_XI_MIN_DB / 10)
    gmin1 = 10 ** (HR_GMIN1_DB / 20)
    gamma = np.minimum(P / N[:, None], 1e4)
    G = np.empty_like(P)
    S_prev = np.zeros_like(N)
    for t in range(P.shape[1]):
        inst = np.maximum(gamma[:, t] - 1, 0)
        xi = inst if t == 0 else HR_ALPHA_DD * S_prev / N + (1 - HR_ALPHA_DD) * inst
        g = np.clip(_hr_lmmse_gain(np.maximum(xi, xi_min), gamma[:, t]), gmin1, 1)
        G[:, t] = g
        S_prev = g ** 2 * P[:, t]
    s = _hr_istft(G * Z, length, n)
    H = _hr_stft(np.maximum(s, 0), n)[:, :Z.shape[1]]
    xi_h = (HR_RHO * np.abs(G * Z) ** 2 + (1 - HR_RHO) * np.abs(H) ** 2) / N[:, None]
    Gh = np.clip(_hr_lmmse_gain(np.maximum(xi_h, xi_min), gamma), 10 ** (HR_GMIN2_DB / 20), 1)
    return _hr_istft(Gh * Z, length, n)

In [17]:
def hrnr_enhance(noisy):
    x = np.asarray(noisy, dtype=np.float64)
    length = len(x)
    if length > 30:
        x = signal.sosfiltfilt(signal.butter(4, HR_HPF_HZ, "highpass", fs=HR_SR, output="sos"), x)
    if len(x) < max(HR_FFTS):
        x = np.pad(x, (0, max(HR_FFTS) - len(x)))
    y = np.mean([_hr_single(x, len(x), n) for n in HR_FFTS], axis=0)
    return y[:length]

In [18]:
logmmse_enhance = hrnr_enhance

In [19]:
def evaluate(df, fn, root=DATA_ROOT):
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df)):
        c, _ = sf.read(root / r.clean_path, dtype="float64")
        y, _ = sf.read(root / r.noisy_path, dtype="float64")
        e = fn(y)
        rows.append({"segment_id": r.segment_id, "noise_category": r.noise_category,
                     "target_snr_db": r.target_snr_db,
                     "pesq_noisy": pesq(16000, c, y, "wb"),
                     "pesq_enh": pesq(16000, c, e, "wb")})
    return pd.DataFrame(rows)

res = evaluate(val_df, hrnr_enhance)
print(res[["pesq_noisy", "pesq_enh"]].mean())
print(res.groupby("target_snr_db")[["pesq_noisy", "pesq_enh"]].mean())
print(res.groupby("noise_category")[["pesq_noisy", "pesq_enh"]].mean().round(3))

  0%|          | 0/164 [00:00<?, ?it/s]

pesq_noisy    1.394672
pesq_enh      1.950599
dtype: float64
               pesq_noisy  pesq_enh
target_snr_db                      
0.0              1.079606  1.291439
5.0              1.218112  1.695708
10.0             1.357503  1.929805
15.0             1.635377  2.320544
                      pesq_noisy  pesq_enh
noise_category                            
aircraft_cockpit           1.173     1.933
alarm_siren                1.311     1.916
animal                     1.423     1.499
bell                       1.675     1.746
broadband_noise            1.115     1.713
clap                       1.378     1.365
click                      2.129     1.989
destroyer_engine           1.248     1.701
destroyer_operations       1.286     2.068
door_moving                1.777     2.103
factory                    1.413     2.090
footsteps                  1.217     1.261
hf_channel                 1.149     1.467
machine                    1.416     1.892
machine_gun                2.085   

In [20]:
def process_folder(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR):
    input_dir, output_dir = Path(input_dir), Path(output_dir)
    files = sorted(p for p in input_dir.rglob("*") if p.suffix.lower() == ".wav")
    for p in tqdm(files):
        x, sr = sf.read(p, dtype="float64")
        if x.ndim > 1:
            x = x.mean(axis=1)
        y = hrnr_enhance(x)
        out = output_dir / p.relative_to(input_dir)
        out.parent.mkdir(parents=True, exist_ok=True)
        sf.write(out, y, sr, subtype="FLOAT")
    print(len(files), "files ->", output_dir)

if INPUT_DIR.is_dir():
    process_folder()